In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

def load_csv(fname):
    # reuse a generic loader; do not modify files on disk
    return pd.read_csv(fname)

def summarize_df(df, filename, name_col=None):
    # Ensure timestamp exists
    if 'timestamp' not in df.columns:
        raise ValueError(f"File {filename} missing 'timestamp' column")
    # Ensure timestamp is numeric
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    total_rows = int(df.shape[0])
    # Compute min/max timestamp as ints (or None if empty)
    if total_rows == 0:
        tmin = None
        tmax = None
        tmin_str = None
        tmax_str = None
    else:
        tmin = int(df['timestamp'].min())
        tmax = int(df['timestamp'].max())
        # convert to UTC then to Asia/Shanghai for human readable
        tmin_str = pd.to_datetime(tmin, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')
        tmax_str = pd.to_datetime(tmax, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')
    # unique cmdb_id values (first 20 alphabetically)
    if 'cmdb_id' in df.columns:
        cmdbs = sorted(df['cmdb_id'].dropna().astype(str).unique())[:20]
    else:
        cmdbs = []
    # unique KPI/trace/log names depending on file
    if name_col and name_col in df.columns:
        names = sorted(df[name_col].dropna().astype(str).unique())[:50]
    else:
        names = []
    # top 5 rows ordered by timestamp ascending, include a compact set of columns
    sample = []
    if total_rows > 0:
        df_sorted = df.sort_values('timestamp', ascending=True)
        # prepare a display DataFrame with selected columns
        display_cols = ['timestamp', 'cmdb_id']
        if name_col and name_col in df.columns:
            display_cols.append(name_col)
        if 'value' in df.columns:
            display_cols.append('value')
        if 'message' in df.columns and 'message' not in display_cols:
            display_cols.append('message')
        # create tz-converted timestamp string column
        df_sorted = df_sorted.copy()
        df_sorted['ts_utc8'] = pd.to_datetime(df_sorted['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
        display_cols = ['timestamp', 'ts_utc8'] + [c for c in display_cols if c not in ['timestamp']]
        sample_df = df_sorted[display_cols].head(5)
        # convert to list of plain dicts (JSON-serializable)
        sample = sample_df.where(pd.notnull(sample_df), None).to_dict(orient='records')
    # compose summary dict
    summary = {
        "total_rows": total_rows,
        "min_timestamp": tmin,
        "min_timestamp_utc8": tmin_str,
        "max_timestamp": tmax,
        "max_timestamp_utc8": tmax_str,
        "unique_cmdb_id_first20": cmdbs,
        f"unique_{name_col}_first50" if name_col else "unique_names": names,
        "top5_rows_by_timestamp": sample
    }
    return summary

# Load files (reuse variable name df for memory efficiency across loads)
df = load_csv('metric.csv')
metric_summary = summarize_df(df, 'metric.csv', name_col='kpi_name')

df = load_csv('trace.csv')
trace_summary = summarize_df(df, 'trace.csv', name_col='trace_name')

df = load_csv('log.csv')
log_summary = summarize_df(df, 'log.csv', name_col='log_name')

df = load_csv('error_logs.csv')
# error_logs.csv has no KPI column; pass name_col=None
error_logs_summary = summarize_df(df, 'error_logs.csv', name_col=None)

# aggregate results into a JSON-serializable dict keyed by filename
summary = {
    "metric.csv": metric_summary,
    "trace.csv": trace_summary,
    "log.csv": log_summary,
    "error_logs.csv": error_logs_summary
}

summary

```
Out[1]:
```
```python
summary_text = (
    "Summary of telemetry files (all timestamps shown in UTC+8):\n\n"
    "metric.csv: 147,012 rows from 2022-03-21 04:30:00 to 2022-03-21 04:59:00. "
    "Contains many service and container KPIs (examples: app.grpc.*, app.http.*, many container.* metrics). "
    "Sample top rows (earliest) are adservice JVM/GC metrics at 04:30.\n\n"
    "trace.csv: 24,656 rows from 2022-03-21 04:30:00 to 2022-03-21 04:59:00. "
    "Trace names include inter-service traces (e.g., trace.from_frontend-*, trace.from_checkoutservice-*). "
    "Earliest traces show very small durations and row counts for adservice pods.\n\n"
    "log.csv: 1,628 rows from 2022-03-21 04:30:00 to 2022-03-21 04:59:00. "
    "Log types are primarily 'log.error_count' and 'log.row_count'. Earliest rows show zero errors and varying row_count values for adservice pods.\n\n"
    "error_logs.csv: 576 rows from 2022-03-21 04:30:07 to 2022-03-21 04:59:59. "
    "Error entries come from frontend and some cartservice pods. Earliest messages repeatedly state: "
    "'severity: warning, message: failed to retrieve ads' (front-end pods).\n\n"
    "Overall quick takeaway: the error logs reveal frontend warnings about failing to retrieve ads around 04:30–04:59, "
    "while metrics and traces include adservice and frontend activity (GC, small trace latencies, request counts). "
    "Investigations should focus on frontend → adservice interactions and the adservice pods around the 04:30 window."
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 147012, 'min_timestamp': 1647808200, 'min_timestamp_utc8': '2022-03-21 04:30:00 CST+0800', 'max_timestamp': 1647809940, 'max_timestamp_utc8': '2022-03-21 04:59:00 CST+0800', 'unique_cmdb_id_first20': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'], 'unique_kpi_name_first50': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'container.node-5.container_cpu_cfs_periods', 'container.node-5.container_cpu_cfs_throttled_periods', 'container.node-5.container_cpu_cfs_throttled_seconds', 'container.node-5.container_cpu_load_average_10s', 'container.node-5.container_cpu_system_seconds', 'container.node-5.container_cpu_usage_seconds', 'container.node-5.container_cpu_user_seconds', 'container.node-5.container_file_descriptors', 'container.node-5.container_fs_inodes./dev/vda1', 'container.node-5.container_fs_inodes_free./dev/vda1', 'container.node-5.container_fs_io_current./dev/vda1', 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'container.node-5.container_fs_limit_MB./dev/vda1', 'container.node-5.container_fs_read_seconds./dev/vda1', 'container.node-5.container_fs_reads./dev/vda', 'container.node-5.container_fs_reads./dev/vda1', 'container.node-5.container_fs_reads_MB./dev/vda', 'container.node-5.container_fs_reads_merged./dev/vda1', 'container.node-5.container_fs_sector_reads./dev/vda1', 'container.node-5.container_fs_sector_writes./dev/vda1', 'container.node-5.container_fs_usage_MB./dev/vda1', 'container.node-5.container_fs_write_seconds./dev/vda1', 'container.node-5.container_fs_writes./dev/vda', 'container.node-5.container_fs_writes./dev/vda1', 'container.node-5.container_fs_writes_MB./dev/vda', 'container.node-5.container_fs_writes_merged./dev/vda1', 'container.node-5.container_last_seen', 'container.node-5.container_memory_cache', 'container.node-5.container_memory_failcnt', 'container.node-5.container_memory_failures.container.pgfault', 'container.node-5.container_memory_failures.container.pgmajfault', 'container.node-5.container_memory_failures.hierarchy.pgfault', 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'container.node-5.container_memory_mapped_file', 'container.node-5.container_memory_max_usage_MB', 'container.node-5.container_memory_rss', 'container.node-5.container_memory_swap', 'container.node-5.container_memory_usage_MB', 'container.node-5.container_memory_working_set_MB', 'container.node-5.container_network_receive_MB.eth0', 'container.node-5.container_network_receive_errors.eth0'], 'top5_rows_by_timestamp': [{'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'value': 1.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'value': 1.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'value': 1.6666666666666667}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'value': 63.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy', 'value': 285145547.25}]}, 'trace.csv': {'total_rows': 24656, 'min_timestamp': 1647808200, 'min_timestamp_utc8': '2022-03-21 04:30:00 CST+0800', 'max_timestamp': 1647809940, 'max_timestamp_utc8': '2022-03-21 04:59:00 CST+0800', 'unique_cmdb_id_first20': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0'], 'unique_trace_name_first50': ['trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_recommendationservice-0.duration_mean', 'trace.from_recommendationservice-0.duration_p95', 'trace.from_recommendationservice-0.error_rate', 'trace.from_recommendationservice-0.row_count', 'trace.from_recommendationservice-1.duration_mean', 'trace.from_recommendationservice-1.duration_p95', 'trace.from_recommendationservice-1.error_rate', 'trace.from_recommendationservice-1.row_count', 'trace.from_recommendationservice-2.duration_mean', 'trace.from_recommendationservice-2.duration_p95', 'trace.from_recommendationservice-2.error_rate', 'trace.from_recommendationservice-2.row_count', 'trace.from_recommendationservice2-0.duration_mean', 'trace.from_recommendationservice2-0.duration_p95', 'trace.from_recommendationservice2-0.error_rate', 'trace.from_recommendationservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95'], 'top5_rows_by_timestamp': [{'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-2.duration_mean', 'value': 1.3e-05}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-2.duration_p95', 'value': 1.4e-05}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-2.error_rate', 'value': 0.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-2.row_count', 'value': 8.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice2-0', 'trace_name': 'trace.from_frontend2-0.duration_mean', 'value': 2.6e-05}]}, 'log.csv': {'total_rows': 1628, 'min_timestamp': 1647808200, 'min_timestamp_utc8': '2022-03-21 04:30:00 CST+0800', 'max_timestamp': 1647809940, 'max_timestamp_utc8': '2022-03-21 04:59:00 CST+0800', 'unique_cmdb_id_first20': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1'], 'unique_log_name_first50': ['log.error_count', 'log.row_count'], 'top5_rows_by_timestamp': [{'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-0', 'log_name': 'log.row_count', 'value': 428.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-1', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-1', 'log_name': 'log.row_count', 'value': 38.0}, {'timestamp': 1647808200, 'ts_utc8': '2022-03-21 04:30:00 CST+0800', 'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'value': 0.0}]}, 'error_logs.csv': {'total_rows': 576, 'min_timestamp': 1647808207, 'min_timestamp_utc8': '2022-03-21 04:30:07 CST+0800', 'max_timestamp': 1647809999, 'max_timestamp_utc8': '2022-03-21 04:59:59 CST+0800', 'unique_cmdb_id_first20': ['cartservice-0', 'cartservice-1', 'cartservice-2', 'frontend-0', 'frontend-1', 'frontend-2'], 'unique_names': [], 'top5_rows_by_timestamp': [{'timestamp': 1647808207, 'ts_utc8': '2022-03-21 04:30:07 CST+0800', 'cmdb_id': 'frontend-0', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': 1647808214, 'ts_utc8': '2022-03-21 04:30:14 CST+0800', 'cmdb_id': 'frontend-2', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': 1647808215, 'ts_utc8': '2022-03-21 04:30:15 CST+0800', 'cmdb_id': 'frontend-0', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': 1647808220, 'ts_utc8': '2022-03-21 04:30:20 CST+0800', 'cmdb_id': 'frontend-2', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': 1647808230, 'ts_utc8': '2022-03-21 04:30:30 CST+0800', 'cmdb_id': 'frontend-1', 'message': 'severity: warning, message: failed to retrieve ads'}]}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse / load metric.csv
df = pd.read_csv('metric.csv')

# ensure numeric types for value and timestamp
df['value'] = pd.to_numeric(df.get('value'), errors='coerce')
df['timestamp'] = pd.to_numeric(df.get('timestamp'), errors='coerce').astype('Int64')

# filter to requested cmdb_ids
cmdb_filter = ['adservice','adservice-0','adservice-1','adservice-2','frontend','frontend-0','frontend-1','frontend-2']
df_f = df[df['cmdb_id'].isin(cmdb_filter)].copy()

# group by cmdb_id and kpi_name and compute aggregations across full series
grouped_frontend_adservice_metrics = (
    df_f
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count = ('value', 'count'),
        min_value = ('value', 'min'),
        max_value = ('value', 'max'),
        p95_value = ('value', lambda s: float(s.quantile(0.95)) if s.count()>0 else None),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max')
    )
)

# sort by count descending and take top 20
top20_df = grouped_frontend_adservice_metrics.sort_values('count', ascending=False).head(20)

# normalize values to plain Python types for JSON-serializable output
def _norm(v):
    if pd.isna(v):
        return None
    # pandas nullable Int64 may be returned; cast to python int
    try:
        if isinstance(v, (pd.Timestamp,)):
            return str(v)
        if hasattr(v, 'dtype'):  # numpy types
            return v.item()
    except Exception:
        pass
    # fallback
    return v

top20_json = [{k: _norm(v) for k, v in row.items()} for row in top20_df.to_dict(orient='records')]

# display the JSON-serializable top20 result (grouped_frontend_adservice_metrics is retained in kernel)
top20_json

```
Out[2]:
```
summary_text = (
    "Summary of grouped metric results for frontend & adservice (UTC+8):\n\n"
    "- Time window: 2022-03-21 04:30:00 to 2022-03-21 04:59:00.\n"
    "- Top groups (20 shown) all have 30 samples (full coverage across the window).\n\n"
    "Notable groups and stats (cmdb_id, kpi_name, count, min, max, P95):\n"
    "1) frontend-2 · mesh.source.frontend.shippingservice.istio_response_messages — 30 samples, min 0.0, max 13.5, P95 ≈ 12.325\n"
    "2) adservice · app.grpc.count — 30 samples, min 13.0, max 99.0, P95 ≈ 97.2\n"
    "3) adservice · app.grpc.mrt — 30 samples, min 0.00182s, max 0.00267s, P95 ≈ 0.00264s\n"
    "4) adservice · app.grpc.rr — 30 samples, constant 100.0 (P95 100.0)\n"
    "5) adservice · app.grpc.sr — 30 samples, min ~53.8, max ~84.2, P95 ≈ 82.64\n"
    "6) adservice · app.http.count — 30 samples, constant 24.0 (P95 24.0)\n"
    "7) adservice · app.http.mrt — 30 samples, P95 ≈ 0.00992s\n"
    "8) multiple adservice runtime/java metrics are stable (e.g., LoadedClassCount = 4990, Compilation_TotalCompilationTime ≈ 43714–43725)\n\n"
    "Quick interpretation:\n"
    "- The adservice shows high gRPC request counts with P95 values close to the observed maxima, indicating sustained high gRPC traffic during the window.\n"
    "- frontend-2 shows elevated Istio response message counts (P95 ~12.3), suggesting frontend-2 is serving notable request volume toward other services.\n"
    "- Many JVM/runtime metrics for adservice are constant/stable in this sample.\n\n"
    "Recommended next steps:\n"
    "- Investigate adservice gRPC traffic patterns and originating frontends (frontend-2 looks prominent).\n"
    "- Correlate these metric groups with trace and error logs for the same time window to see if high traffic corresponds to errors or latency spikes."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'frontend-2', 'kpi_name': 'mesh.source.frontend.shippingservice.istio_response_messages', 'count': 30, 'min_value': 0.0, 'max_value': 13.5, 'p95_value': 12.324999999999996, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.count', 'count': 30, 'min_value': 13.0, 'max_value': 99.0, 'p95_value': 97.19999999999999, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.mrt', 'count': 30, 'min_value': 0.0018222222222248, 'max_value': 0.0026727272727263, 'p95_value': 0.002640093984965935, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.rr', 'count': 30, 'min_value': 100.0, 'max_value': 100.0, 'p95_value': 100.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'count': 30, 'min_value': 53.84615384615385, 'max_value': 84.21052631578947, 'p95_value': 82.64057971014493, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.count', 'count': 30, 'min_value': 24.0, 'max_value': 24.0, 'p95_value': 24.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.mrt', 'count': 30, 'min_value': 0.0090979166666656, 'max_value': 0.0133416666666647, 'p95_value': 0.009921979166666324, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.rr', 'count': 30, 'min_value': 100.0, 'max_value': 100.0, 'p95_value': 100.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.sr', 'count': 30, 'min_value': 100.0, 'max_value': 100.0, 'p95_value': 100.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_ClassLoading_LoadedClassCount', 'count': 30, 'min_value': 4990.0, 'max_value': 4990.0, 'p95_value': 4990.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'count': 30, 'min_value': 4990.0, 'max_value': 4990.0, 'p95_value': 4990.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'count': 30, 'min_value': 0.0, 'max_value': 0.0, 'p95_value': 0.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_ClassLoading_Verbose', 'count': 30, 'min_value': 0.0, 'max_value': 0.0, 'p95_value': 0.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'count': 30, 'min_value': 1.0, 'max_value': 1.0, 'p95_value': 1.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_Compilation_TotalCompilationTime', 'count': 30, 'min_value': 43714.0, 'max_value': 43725.0, 'p95_value': 43725.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'count': 30, 'min_value': 42144.833333333336, 'max_value': 42391.16666666666, 'p95_value': 42378.8875, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'count': 30, 'min_value': 3.0, 'max_value': 3.0, 'p95_value': 3.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'frontend-2', 'kpi_name': 'mesh.source.frontend.checkoutservice.istio_request_messages', 'count': 30, 'min_value': 0.0, 'max_value': 2.5, 'p95_value': 2.5, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'frontend-2', 'kpi_name': 'mesh.source.frontend.checkoutservice.istio_requests.grpc.200.0.0', 'count': 30, 'min_value': 0.0, 'max_value': 2.5, 'p95_value': 2.5, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}, {'cmdb_id': 'frontend-2', 'kpi_name': 'mesh.source.frontend.checkoutservice.istio_response_bytes.grpc.200.0.0', 'count': 30, 'min_value': 0.0, 'max_value': 887.5, 'p95_value': 883.0, 'min_timestamp': 1647808200, 'max_timestamp': 1647809940}]```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df if present; otherwise load metric.csv
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

# Ensure correct dtypes
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Target cmdb_ids
cmdb_filter = ['adservice','adservice-0','adservice-1','adservice-2','frontend','frontend-0','frontend-1','frontend-2']

# Filter to relevant components (for threshold computation and later window filtering)
df_sel = df[df['cmdb_id'].isin(cmdb_filter)].copy()

# Rule 9: compute global thresholds (P95 and P5) per (cmdb_id, kpi_name) using the entire available series BEFORE window filtering
thr = (
    df_sel
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        p95_value = ('value', lambda s: float(s.quantile(0.95))),
        p5_value  = ('value', lambda s: float(s.quantile(0.05))),
        series_count = ('value', 'count')
    )
)

# Define incident window
start_ts = 1647808200
end_ts = 1647809940

# Filter rows to the incident window (inclusive)
df_window = df_sel[(df_sel['timestamp'] >= start_ts) & (df_sel['timestamp'] <= end_ts)].copy()

# If no rows in window, produce empty summary
if df_window.shape[0] == 0:
    anomalies_frontend_adservice = pd.DataFrame(columns=df_sel.columns.tolist() + ['p95_value','p5_value','is_anomaly','threshold_direction'])
    summary_df = pd.DataFrame(columns=['cmdb_id','kpi_name','p95_value','window_point_count','anomalies_count','earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window'])
else:
    # Attach thresholds to window rows
    df_window = df_window.merge(thr, on=['cmdb_id','kpi_name'], how='left')

    # Determine whether a KPI is "lower-is-bad" based on name containing 'sr' or 'success' or 'rate' (case-insensitive)
    df_window['lower_is_bad'] = df_window['kpi_name'].str.contains(r'(?i)(sr|success|rate)', regex=True)

    # Determine anomaly per row:
    # - if lower_is_bad: anomaly when value <= p5_value
    # - else: anomaly when value >= p95_value
    # Also guard against NaNs in thresholds/values
    df_window['is_anomaly'] = False
    # apply conditions
    mask_lower = df_window['lower_is_bad'] & df_window['value'].notna() & df_window['p5_value'].notna() & (df_window['value'] <= df_window['p5_value'])
    mask_upper = (~df_window['lower_is_bad']) & df_window['value'].notna() & df_window['p95_value'].notna() & (df_window['value'] >= df_window['p95_value'])
    df_window.loc[mask_lower | mask_upper, 'is_anomaly'] = True
    df_window['threshold_direction'] = df_window['lower_is_bad'].map({True: 'le_p5', False: 'ge_p95'})

    # Store full anomaly rows for later inspection (kernel variable)
    anomalies_frontend_adservice = df_window[df_window['is_anomaly']].copy()

    # Compute per-group aggregates required for summary
    window_counts = df_window.groupby(['cmdb_id','kpi_name'], as_index=False).agg(window_point_count=('value','count'),
                                                                                  max_value_in_window=('value','max'))
    anomaly_agg = anomalies_frontend_adservice.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        anomalies_count = ('is_anomaly','count'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        latest_anomaly_timestamp = ('timestamp','max')
    )

    # Combine thresholds, window counts, anomalies
    summary_df = (
        thr[['cmdb_id','kpi_name','p95_value']]
        .merge(window_counts, on=['cmdb_id','kpi_name'], how='left')
        .merge(anomaly_agg, on=['cmdb_id','kpi_name'], how='left')
    )

    # Fill NaNs for groups that had no window rows or no anomalies
    summary_df['window_point_count'] = summary_df['window_point_count'].fillna(0).astype(int)
    summary_df['anomalies_count'] = summary_df['anomalies_count'].fillna(0).astype(int)
    # earliest/latest timestamps keep NaN for groups without anomalies
    summary_df['earliest_anomaly_timestamp'] = summary_df['earliest_anomaly_timestamp'].astype('Int64')
    summary_df['latest_anomaly_timestamp'] = summary_df['latest_anomaly_timestamp'].astype('Int64')
    summary_df['max_value_in_window'] = summary_df['max_value_in_window'].where(pd.notna(summary_df['max_value_in_window']), None)

    # Keep only groups that have at least one anomaly in the window
    summary_df = summary_df[summary_df['anomalies_count'] > 0].copy()

    # Order by earliest_anomaly_timestamp ascending and limit to top 20
    summary_df = summary_df.sort_values('earliest_anomaly_timestamp', ascending=True).head(20)

    # Ensure final column ordering per spec
    summary_df = summary_df[['cmdb_id','kpi_name','p95_value','window_point_count','anomalies_count','earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window']]

# Keep anomalies_frontend_adservice and summary_df in kernel, display summary_df (compact)
summary_df

```
Out[3]:
```
Summary of anomaly detection for frontend & adservice (window 2022-03-21 04:30:00 — 04:59:00, timestamps in UTC+8):

- Result storage: per-point anomaly rows are saved in the kernel variable anomalies_frontend_adservice for further inspection. The returned summary lists only grouped summaries.

- Overall: multiple (cmdb_id, kpi_name) groups showed anomalies within the incident window. Many groups have anomalies starting at the earliest window timestamp (1647808200).

- Notable findings:
  - adservice-2: many mesh-related KPIs (many "mesh.destination.*" and "mesh.source.*" metrics) have p95 = 0.0 and were flagged as anomalous for all 30 window points (anomalies_count = 30, max_value_in_window = 0.0). This indicates those mesh metrics were consistently zero in the window (which triggered the >= P95 rule).
  - frontend (app.http.rr): p95 = 100.0, window_point_count = 30, anomalies_count = 26, earliest anomaly 1647808200, latest 1647809940, max_value_in_window = 100.0. (High/consistent values near the P95.)
  - frontend-0 container CPU and filesystem KPIs:
    - container.node-5.container_cpu_system_seconds (p95 = 0.05): 30 points in window, 3 anomalies between 1647808200 and 1647808320, max 0.065.
    - container.node-5.container_cpu_usage_seconds (p95 ≈ 0.1584): 30 points, 2 anomalies early in the window, max ≈ 0.1818.
    - container file descriptors, inode, fs io metrics: several groups show p95 = 0.0 and anomalies_count = 30 (all-zero behavior).
  
- Interpretation / quick guidance:
  - The clusters of mesh metrics on adservice-2 being zero across the window is suspicious (could indicate missing traffic, instrumentation gaps, or genuine zero traffic). Because thresholds were computed globally (per spec), p95=0 caused zeros to be classified as anomalies — check whether zero is expected or indicates metric collection/mesh issues.
  - frontend app.http.rr showing many points at/near 100 suggests consistently high success/response rate for those requests — investigate whether this is expected or masking other issues.
  - A few CPU-related metrics on frontend-0 exceeded their P95 briefly at the start of the window; these are lower-volume anomalies but worth correlating with traces/logs.

- Next steps:
  - Inspect anomalies_frontend_adservice (per-point rows) to see exact timestamps and values; correlate with trace.csv and error_logs.csv for the same timestamps to determine impact and root cause.
  - Verify whether p95=0 for many mesh KPIs reflects valid behavior or telemetry/collection issues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name   p95_value  window_point_count  anomalies_count  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window
607   frontend-0      container.node-5.container_cpu_system_seconds    0.050000                  30                3                  1647808200                1647808320             0.065000
584  adservice-2  mesh.destination.frontend.adservice.istio_requ...    0.000000                  30               30                  1647808200                1647809940             0.000000
586  adservice-2  mesh.destination.frontend.adservice.istio_requ...    0.000000                  30               30                  1647808200                1647809940             0.000000
587  adservice-2  mesh.destination.frontend.adservice.istio_resp...    0.000000                  30               30                  1647808200                1647809940             0.000000
589  adservice-2  mesh.destination.frontend.adservice.istio_resp...    0.000000                  30               30                  1647808200                1647809940             0.000000
591  adservice-2  mesh.source.adservice.basic-tidb.istio_tcp_con...    0.000000                  30               30                  1647808200                1647809940             0.000000
592  adservice-2  mesh.source.adservice.basic-tidb.istio_tcp_con...    0.000000                  30               30                  1647808200                1647809940             0.000000
593  adservice-2  mesh.source.adservice.basic-tidb.istio_tcp_rec...    0.000000                  30               30                  1647808200                1647809940             0.000000
594  adservice-2  mesh.source.adservice.basic-tidb.istio_tcp_sen...    0.000000                  30               30                  1647808200                1647809940             0.000000
601     frontend                                        app.http.rr  100.000000                  30               26                  1647808200                1647809940           100.000000
604   frontend-0  container.node-5.container_cpu_cfs_throttled_p...    0.000000                  30               30                  1647808200                1647809940             0.000000
605   frontend-0  container.node-5.container_cpu_cfs_throttled_s...    0.000000                  30               30                  1647808200                1647809940             0.000000
606   frontend-0    container.node-5.container_cpu_load_average_10s    0.000000                  30               30                  1647808200                1647809940             0.000000
582  adservice-2  mesh.destination.frontend.adservice.istio_requ...    0.000000                  30               30                  1647808200                1647809940             0.000000
608   frontend-0       container.node-5.container_cpu_usage_seconds    0.158421                  30                2                  1647808200                1647808260             0.181833
610   frontend-0        container.node-5.container_file_descriptors   17.000000                  30                6                  1647808200                1647808500            17.000000
611   frontend-0     container.node-5.container_fs_inodes./dev/vda1    0.000000                  30               30                  1647808200                1647809940             0.000000
612   frontend-0  container.node-5.container_fs_inodes_free./dev...    0.000000                  30               30                  1647808200                1647809940             0.000000
613   frontend-0  container.node-5.container_fs_io_current./dev/...    0.000000                  30               30                  1647808200                1647809940             0.000000
614   frontend-0  container.node-5.container_fs_io_time_seconds....    0.000000                  30               30                  1647808200                1647809940             0.000000```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse variables where possible; load trace.csv
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure dtypes
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Incident window
start_ts = 1647808200
end_ts = 1647809940

# Filter criteria
cmdb_keep = ['adservice','adservice-0','adservice-1','adservice-2','adservice2-0']
mask_window = (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
mask_name = df_trace['trace_name'].str.contains('from_frontend', na=False)  # case-sensitive
mask_cmdb = df_trace['cmdb_id'].isin(cmdb_keep)

traces_frontend_to_adservice = df_trace[mask_window & mask_name & mask_cmdb].copy()

# Prepare aggregation per (cmdb_id, trace_name)
rows = []
for (cmdb_id, trace_name), g in traces_frontend_to_adservice.groupby(['cmdb_id','trace_name'], sort=True):
    cnt = int(g.shape[0])
    # initialize fields
    row_count_min = row_count_max = None
    duration_mean_min = duration_mean_max = None
    duration_p95_min = duration_p95_max = None
    error_rate_min = error_rate_max = None
    earliest_error_rate_ts = pd.NA
    earliest_zero_rowcount_ts = pd.NA

    if trace_name.endswith('.row_count'):
        row_count_min = float(g['value'].min()) if not g['value'].isna().all() else None
        row_count_max = float(g['value'].max()) if not g['value'].isna().all() else None
        # earliest zero row_count in window
        zeros = g[g['value'] == 0]
        if not zeros.empty:
            earliest_zero_rowcount_ts = int(zeros['timestamp'].min())

    if trace_name.endswith('.duration_mean'):
        duration_mean_min = float(g['value'].min()) if not g['value'].isna().all() else None
        duration_mean_max = float(g['value'].max()) if not g['value'].isna().all() else None

    if trace_name.endswith('.duration_p95'):
        duration_p95_min = float(g['value'].min()) if not g['value'].isna().all() else None
        duration_p95_max = float(g['value'].max()) if not g['value'].isna().all() else None

    if trace_name.endswith('.error_rate'):
        error_rate_min = float(g['value'].min()) if not g['value'].isna().all() else None
        error_rate_max = float(g['value'].max()) if not g['value'].isna().all() else None
        # earliest nonzero error_rate in window
        nonzero = g[g['value'] > 0.0]
        if not nonzero.empty:
            earliest_error_rate_ts = int(nonzero['timestamp'].min())

    # earliest anomaly timestamp: min of earliest_error_rate_ts and earliest_zero_rowcount_ts (ignore NA)
    candidate_ts = []
    if pd.notna(earliest_error_rate_ts):
        candidate_ts.append(int(earliest_error_rate_ts))
    if pd.notna(earliest_zero_rowcount_ts):
        candidate_ts.append(int(earliest_zero_rowcount_ts))
    earliest_anomaly_ts = min(candidate_ts) if candidate_ts else pd.NA

    rows.append({
        'cmdb_id': cmdb_id,
        'trace_name': trace_name,
        'count': cnt,
        'row_count_min': row_count_min,
        'row_count_max': row_count_max,
        'duration_mean_min': duration_mean_min,
        'duration_mean_max': duration_mean_max,
        'duration_p95_min': duration_p95_min,
        'duration_p95_max': duration_p95_max,
        'error_rate_min': error_rate_min,
        'error_rate_max': error_rate_max,
        'earliest_error_rate_ts': earliest_error_rate_ts,
        'earliest_zero_rowcount_ts': earliest_zero_rowcount_ts,
        'earliest_anomaly_ts': earliest_anomaly_ts
    })

summary_df = pd.DataFrame(rows)

# Keep only groups that have at least one detected anomaly timestamp (per instruction ordering)
summary_df = summary_df[pd.notna(summary_df['earliest_anomaly_ts'])].copy()

# Order by earliest detected anomaly timestamp ascending and limit to top 20
summary_df = summary_df.sort_values('earliest_anomaly_ts', ascending=True).head(20)

# Keep columns compact and cast timestamp columns to Int64 where present
for c in ['earliest_error_rate_ts','earliest_zero_rowcount_ts','earliest_anomaly_ts']:
    if c in summary_df.columns:
        summary_df[c] = summary_df[c].astype('Int64')

# Display the compact summary (up to 20 rows) and keep traces_frontend_to_adservice in the kernel
summary_df.reset_index(drop=True).head(20)

```
Out[4]:
```
summary = (
    "Summary (plain English):\n\n"
    "- I filtered trace.csv to the incident window 1647808200–1647809940 (UTC+8: 2022-03-21 04:30–04:59) and to trace_name containing 'from_frontend' "
    "and cmdb_id in ['adservice','adservice-0','adservice-1','adservice-2','adservice2-0'].\n\n"
    "- Result: no matching trace rows were found (traces_frontend_to_adservice is empty). Therefore there are no per-(cmdb_id, trace_name) groups to aggregate and no detected trace anomalies in that filtered set.\n\n"
    "- Suggested next steps: verify correct trace_name patterns (case-sensitive 'from_frontend'), confirm the target cmdb_id values exist in trace.csv, or broaden the filter (e.g., remove case sensitivity or search for 'frontend' more generally) to locate relevant frontend→adservice traces."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, count, row_count_min, row_count_max, duration_mean_min, duration_mean_max, duration_p95_min, duration_p95_max, error_rate_min, error_rate_max, earliest_error_rate_ts, earliest_zero_rowcount_ts, earliest_anomaly_ts]
Index: []```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# --- Part 1: error_logs.csv frontend filtering and summaries ---

# Load error logs (reuse variable name if present)
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv')

# normalize types
df_err['timestamp'] = pd.to_numeric(df_err['timestamp'], errors='coerce').astype('Int64')
df_err['cmdb_id'] = df_err['cmdb_id'].astype(str)
df_err['message'] = df_err['message'].astype(str)

# Incident window filter
start_ts = 1647808200
end_ts = 1647809940

mask_window = (df_err['timestamp'] >= start_ts) & (df_err['timestamp'] <= end_ts)
mask_frontend = df_err['cmdb_id'].str.startswith('frontend', na=False)

error_logs_frontend = df_err[mask_window & mask_frontend].copy()  # kernel variable for later inspection

# Aggregation per frontend cmdb_id: count, earliest, latest
grp = (
    error_logs_frontend
    .groupby('cmdb_id', as_index=False)
    .agg(
        error_count=('timestamp','count'),
        earliest_timestamp_num=('timestamp','min'),
        latest_timestamp_num=('timestamp','max')
    )
)

# Convert timestamps to UTC+8 human strings for compact display
grp['earliest_timestamp_utc8'] = pd.to_datetime(grp['earliest_timestamp_num'].astype('Int64'), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
grp['latest_timestamp_utc8'] = pd.to_datetime(grp['latest_timestamp_num'].astype('Int64'), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# Top 5 most frequent message strings per cmdb_id
top_msgs = []
for cmdb, g in error_logs_frontend.groupby('cmdb_id'):
    msg_counts = g['message'].value_counts().reset_index()
    msg_counts.columns = ['message','count']
    top5 = list(msg_counts.head(5).itertuples(index=False, name=None))  # list of (message, count)
    top_msgs.append({'cmdb_id': cmdb, 'top5_messages': top5})

top_msgs_df = pd.DataFrame(top_msgs)

# Merge top messages into grp for compact summary
error_summary = grp.merge(top_msgs_df, on='cmdb_id', how='left')[['cmdb_id','error_count','earliest_timestamp_num','earliest_timestamp_utc8','latest_timestamp_num','latest_timestamp_utc8','top5_messages']]

# Earliest 10 raw error rows that contain 'failed to retrieve ads' (case-sensitive as given)
mask_failed_ads = error_logs_frontend['message'].str.contains('failed to retrieve ads', na=False)
failed_rows_10 = error_logs_frontend[mask_failed_ads].sort_values('timestamp', ascending=True).head(10).copy()
if not failed_rows_10.empty:
    failed_rows_10['timestamp_utc8'] = pd.to_datetime(failed_rows_10['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    failed_rows_10 = failed_rows_10[['timestamp','timestamp_utc8','cmdb_id','message']]

# --- Part 2: adservice metrics in-window summaries ---

# Load metric.csv (reuse df if present)
try:
    df_metrics
except NameError:
    df_metrics = pd.read_csv('metric.csv')

df_metrics['timestamp'] = pd.to_numeric(df_metrics['timestamp'], errors='coerce').astype('Int64')
df_metrics['value'] = pd.to_numeric(df_metrics['value'], errors='coerce')
df_metrics['cmdb_id'] = df_metrics['cmdb_id'].astype(str)
df_metrics['kpi_name'] = df_metrics['kpi_name'].astype(str)

# Filter to adservice service and pods and window
ad_cmdbs = ['adservice','adservice-0','adservice-1','adservice-2']
mask_ad = df_metrics['cmdb_id'].isin(ad_cmdbs)
mask_window_m = (df_metrics['timestamp'] >= start_ts) & (df_metrics['timestamp'] <= end_ts)

df_ad_window = df_metrics[mask_ad & mask_window_m].copy()
adservice_metrics_window = df_ad_window.copy()  # kernel variable for later inspection

# Select KPIs of interest:
kpi_explicit = ['app.grpc.count','app.grpc.sr','app.grpc.mrt','app.http.count','app.http.sr']
mask_kpi = df_ad_window['kpi_name'].isin(kpi_explicit) | df_ad_window['kpi_name'].str.contains(r'(?i)(error|err)', regex=True)
df_ad_selected = df_ad_window[mask_kpi].copy()

# Aggregations per (cmdb_id, kpi_name) across the window
metrics_agg = (
    df_ad_selected
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        count=('value','count'),
        min_value=('value','min'),
        max_value=('value','max'),
        mean_value=('value','mean'),
        earliest_timestamp_num=('timestamp','min')
    )
)

# Add human-readable earliest timestamp in UTC+8
metrics_agg['earliest_timestamp_utc8'] = pd.to_datetime(metrics_agg['earliest_timestamp_num'].astype('Int64'), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# Up to 20 sample rows ordered by timestamp ascending
adservice_metrics_window_sample = df_ad_selected.sort_values('timestamp', ascending=True)[['timestamp','cmdb_id','kpi_name','value']].head(20).copy()
if not adservice_metrics_window_sample.empty:
    adservice_metrics_window_sample['timestamp_utc8'] = pd.to_datetime(adservice_metrics_window_sample['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    adservice_metrics_window_sample = adservice_metrics_window_sample[['timestamp','timestamp_utc8','cmdb_id','kpi_name','value']]

# --- Final compact outputs (kept small) ---
# error_summary: per-frontend aggregated counts + top5 messages
# failed_rows_10: earliest 10 raw error rows containing 'failed to retrieve ads' (if any)
# metrics_agg: adservice KPI aggregates (per group)
# adservice_metrics_window_sample: up to 20 sample rows

error_summary, failed_rows_10.head(10) if 'failed_rows_10' in locals() else pd.DataFrame(), metrics_agg.head(20), adservice_metrics_window_sample.head(20)

```
Out[5]:
```
Summary (plain English):

Part 1 — frontend error logs (window 2022-03-21 04:30:00 — 04:59:00 UTC+8)
- I filtered error_logs.csv to frontend components and the incident window; results are saved in kernel variable error_logs_frontend.
- Counts and time range per frontend:
  - frontend-0: 38 error rows, earliest 2022-03-21 04:30:07, latest 2022-03-21 04:36:31.
  - frontend-1: 230 error rows, earliest 2022-03-21 04:30:30, latest 2022-03-21 04:58:55.
  - frontend-2: 248 error rows, earliest 2022-03-21 04:30:14, latest 2022-03-21 04:58:58.
- Top message(s): the most frequent message strings are warnings like "severity: warning, message: failed to retrieve ads" (this appears repeatedly across frontends).
- I also extracted the earliest 10 raw error rows that contain the substring "failed to retrieve ads" (shown in the result) for direct inspection.

Part 2 — adservice metrics in-window (same window)
- I filtered metric.csv to adservice (service) and adservice pods and selected KPIs of interest; the filtered rows are saved in kernel variable adservice_metrics_window.
- Aggregated metrics (per cmdb_id, kpi_name) across the window (examples from output):
  - adservice · app.grpc.count — 30 points, min 13, max 99, mean ≈ 69.97, earliest 2022-03-21 04:30:00.
  - adservice · app.grpc.mrt — 30 points, min ≈ 0.001822 s, max ≈ 0.002673 s, mean ≈ 0.002475 s.
  - adservice · app.grpc.sr — 30 points, min ≈ 53.85, max ≈ 84.21, mean ≈ 77.61.
  - adservice · app.http.count — 30 points, constant 24.
  - adservice · app.http.sr — 30 points, constant 100.
- Many container/mesh-related KPIs for adservice-0/1/2 show counts = 30 and values = 0 (e.g., container network receive/transmit errors and some fs/inode metrics) — these zeroed metrics appear across pods in the window.
- A small sample (up to 20 rows) ordered by timestamp was returned (earliest rows show app.grpc/app.http metrics at 04:30 and many container metrics with value 0).

Quick interpretation & recommended next steps
- Frontend error logs strongly indicate repeated "failed to retrieve ads" warnings across frontend-0/1/2 beginning ~04:30. This is likely user-impacting.
- Adservice shows sustained gRPC traffic (counts high, mean ~70, some peaks to ~99) with generally high success rates; app.http counts are steady. At first glance adservice appears to be receiving traffic and serving requests.
- The mesh/container metrics having many zeros (especially for adservice-2 mesh metrics) is suspicious — it could indicate missing telemetry, mesh instrumentation gaps, or genuine zero traffic on certain mesh metrics. Because error logs show frontend failures retrieving ads, check connectivity between frontend → adservice (and the adservice pods).
- Next steps: correlate timestamps from error_logs_frontend with adservice_metrics_window and traces (if traces exist) to determine whether frontends failed due to adservice behavior, network/mesh issues, or telemetry gaps. Inspect the saved kernel variables error_logs_frontend and adservice_metrics_window for per-point detail.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  error_count  earliest_timestamp_num       earliest_timestamp_utc8  latest_timestamp_num         latest_timestamp_utc8                                      top5_messages
0  frontend-0           38              1647808207  2022-03-21 04:30:07 CST+0800            1647808591  2022-03-21 04:36:31 CST+0800  [(severity: warning, message: failed to retrie...
1  frontend-1          230              1647808230  2022-03-21 04:30:30 CST+0800            1647809935  2022-03-21 04:58:55 CST+0800  [(severity: warning, message: failed to retrie...
2  frontend-2          248              1647808214  2022-03-21 04:30:14 CST+0800            1647809938  2022-03-21 04:58:58 CST+0800  [(severity: warning, message: failed to retrie...,     timestamp                timestamp_utc8     cmdb_id                                            message
0  1647808207  2022-03-21 04:30:07 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
1  1647808214  2022-03-21 04:30:14 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
2  1647808215  2022-03-21 04:30:15 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
3  1647808220  2022-03-21 04:30:20 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
6  1647808230  2022-03-21 04:30:30 CST+0800  frontend-1  severity: warning, message: failed to retrieve...
4  1647808230  2022-03-21 04:30:30 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
5  1647808230  2022-03-21 04:30:30 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
7  1647808235  2022-03-21 04:30:35 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
8  1647808237  2022-03-21 04:30:37 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
9  1647808239  2022-03-21 04:30:39 CST+0800  frontend-1  severity: warning, message: failed to retrieve...,         cmdb_id                                           kpi_name  count   min_value   max_value  mean_value  earliest_timestamp_num       earliest_timestamp_utc8
0     adservice                                     app.grpc.count     30   13.000000   99.000000   69.966667              1647808200  2022-03-21 04:30:00 CST+0800
1     adservice                                       app.grpc.mrt     30    0.001822    0.002673    0.002475              1647808200  2022-03-21 04:30:00 CST+0800
2     adservice                                        app.grpc.sr     30   53.846154   84.210526   77.607186              1647808200  2022-03-21 04:30:00 CST+0800
3     adservice                                     app.http.count     30   24.000000   24.000000   24.000000              1647808200  2022-03-21 04:30:00 CST+0800
4     adservice                                        app.http.sr     30  100.000000  100.000000  100.000000              1647808200  2022-03-21 04:30:00 CST+0800
5   adservice-0  container.node-5.container_network_receive_err...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800
6   adservice-0  container.node-5.container_network_transmit_er...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800
7   adservice-0  container.node-5.container_tasks_state.uninter...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800
8   adservice-1  container.node-5.container_network_receive_err...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800
9   adservice-1  container.node-5.container_network_transmit_er...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800
10  adservice-1  container.node-5.container_tasks_state.uninter...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800
11  adservice-2  container.node-5.container_network_receive_err...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800
12  adservice-2  container.node-5.container_network_transmit_er...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800
13  adservice-2  container.node-5.container_tasks_state.uninter...     30    0.000000    0.000000    0.000000              1647808200  2022-03-21 04:30:00 CST+0800,        timestamp                timestamp_utc8      cmdb_id                                           kpi_name       value
0     1647808200  2022-03-21 04:30:00 CST+0800    adservice                                     app.grpc.count   69.000000
1     1647808200  2022-03-21 04:30:00 CST+0800    adservice                                       app.grpc.mrt    0.002530
3     1647808200  2022-03-21 04:30:00 CST+0800    adservice                                        app.grpc.sr   72.463768
4     1647808200  2022-03-21 04:30:00 CST+0800    adservice                                     app.http.count   24.000000
554   1647808200  2022-03-21 04:30:00 CST+0800  adservice-2  container.node-5.container_network_receive_err...    0.000000
487   1647808200  2022-03-21 04:30:00 CST+0800  adservice-1  container.node-5.container_tasks_state.uninter...    0.000000
401   1647808200  2022-03-21 04:30:00 CST+0800  adservice-0  container.node-5.container_tasks_state.uninter...    0.000000
386   1647808200  2022-03-21 04:30:00 CST+0800  adservice-0  container.node-5.container_network_transmit_er...    0.000000
573   1647808200  2022-03-21 04:30:00 CST+0800  adservice-2  container.node-5.container_tasks_state.uninter...    0.000000
558   1647808200  2022-03-21 04:30:00 CST+0800  adservice-2  container.node-5.container_network_transmit_er...    0.000000
382   1647808200  2022-03-21 04:30:00 CST+0800  adservice-0  container.node-5.container_network_receive_err...    0.000000
7     1647808200  2022-03-21 04:30:00 CST+0800    adservice                                        app.http.sr  100.000000
472   1647808200  2022-03-21 04:30:00 CST+0800  adservice-1  container.node-5.container_network_transmit_er...    0.000000
468   1647808200  2022-03-21 04:30:00 CST+0800  adservice-1  container.node-5.container_network_receive_err...    0.000000
4878  1647808260  2022-03-21 04:31:00 CST+0800    adservice                                     app.http.count   24.000000
4877  1647808260  2022-03-21 04:31:00 CST+0800    adservice                                        app.grpc.sr   77.777778
5256  1647808260  2022-03-21 04:31:00 CST+0800  adservice-0  container.node-5.container_network_receive_err...    0.000000
4881  1647808260  2022-03-21 04:31:00 CST+0800    adservice                                        app.http.sr  100.000000
4875  1647808260  2022-03-21 04:31:00 CST+0800    adservice                                       app.grpc.mrt    0.002493
4874  1647808260  2022-03-21 04:31:00 CST+0800    adservice                                     app.grpc.count   99.000000)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse existing df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# ensure dtypes
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# incident window
start_ts = 1647808200
end_ts = 1647809940

# filter rows where trace_name contains 'from_frontend' (case-sensitive) and within window
mask_window = (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
mask_name = df_trace['trace_name'].str.contains('from_frontend', na=False)  # case-sensitive
traces_frontend_alltargets = df_trace[mask_window & mask_name].copy()  # kernel variable for later inspection

# compute unique cmdb_id list (alphabetical, first 20)
unique_cmdb_ids = sorted(traces_frontend_alltargets['cmdb_id'].dropna().astype(str).unique())[:20]

# group by (trace_name, cmdb_id) and compute aggregates
grouped = (
    traces_frontend_alltargets
    .groupby(['trace_name','cmdb_id'], as_index=False)
    .agg(
        count = ('value','count'),
        min_value = ('value','min'),
        max_value = ('value','max'),
        mean_value = ('value','mean'),
        earliest_timestamp = ('timestamp','min'),
        latest_timestamp = ('timestamp','max')
    )
)

# add human-readable UTC+8 timestamps for earliest/latest (compact)
if not grouped.empty:
    grouped['earliest_timestamp_utc8'] = pd.to_datetime(grouped['earliest_timestamp'].astype('Int64'), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    grouped['latest_timestamp_utc8']   = pd.to_datetime(grouped['latest_timestamp'].astype('Int64'), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# order by count descending and limit to top 20 groups
group_summary = grouped.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Keep output compact: selected columns and up to 20 rows
cols_out = ['trace_name','cmdb_id','count','min_value','max_value','mean_value','earliest_timestamp','earliest_timestamp_utc8','latest_timestamp','latest_timestamp_utc8']
# some columns may be absent if grouped is empty; intersect
cols_out = [c for c in cols_out if c in group_summary.columns]
group_summary = group_summary[cols_out]

# Display the compact summary and the unique cmdb_id list (traces_frontend_alltargets retained in kernel)
group_summary, unique_cmdb_ids

```
Out[6]:
```
Summary (plain English):

- I filtered trace.csv to the incident window (2022-03-21 04:30:00 — 04:59:00 UTC+8) and selected all trace records whose trace_name contains "from_frontend". The filtered rows are saved in the kernel variable traces_frontend_alltargets for further inspection.

- Unique target components observed (first 20): 
  adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice-0, currencyservice-1, currencyservice-2, currencyservice2-0, productcatalogservice-0, productcatalogservice-1, productcatalogservice-2, productcatalogservice2-0.

- Up to 20 top groups by count (all have 30 samples across the window). Notable group summaries:
  - trace.from_frontend2-0.row_count → productcatalogservice2-0: 30 samples, row_count min 8, max 287, mean ≈ 203.77 (earliest 04:30, latest 04:59).
  - trace.from_frontend2-0.row_count → currencyservice2-0: 30 samples, min 8, max 208, mean ≈ 129.93.
  - Many trace.from_frontend*.error_rate groups across multiple targets show error_rate = 0.0 for the whole window (count 30, min/max/mean = 0.0).
  - trace.from_frontend2-0.duration_p95 → productcatalogservice2-0: 30 samples, duration_p95 min ≈ 0.00494s, max ≈ 0.00587s, mean ≈ 0.00531s.
  - Multiple trace.from_frontend-1.duration_mean groups for productcatalogservice-* have duration_mean in the ~0.0036–0.0045s range (stable across the window).

- Interpretation:
  - Frontend→many downstream services show consistent traffic (30 samples each across the 30-minute window).
  - Row counts to productcatalogservice and currencyservice targets are relatively high (means shown above).
  - Error rates for the observed from_frontend traces are essentially zero in this window (no obvious trace-level errors reported).
  - Latencies (duration_mean/duration_p95) are small and fairly stable for the listed targets.

- Next steps:
  - Inspect traces_frontend_alltargets (saved in the kernel) if you need per-point timestamps or want to correlate these trace metrics with the frontend error logs and adservice metrics already collected.

The original code execution output of IPython Kernel is also provided below for reference:

(                             trace_name                   cmdb_id  count  min_value   max_value  mean_value  earliest_timestamp       earliest_timestamp_utc8  latest_timestamp         latest_timestamp_utc8
0      trace.from_frontend2-0.row_count  productcatalogservice2-0     30   8.000000  287.000000  203.766667          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
1      trace.from_frontend2-0.row_count        currencyservice2-0     30   8.000000  208.000000  129.933333          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
2     trace.from_frontend2-0.error_rate  productcatalogservice2-0     30   0.000000    0.000000    0.000000          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
3     trace.from_frontend2-0.error_rate        currencyservice2-0     30   0.000000    0.000000    0.000000          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
4      trace.from_frontend-2.error_rate   productcatalogservice-1     30   0.000000    0.000000    0.000000          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
5   trace.from_frontend2-0.duration_p95  productcatalogservice2-0     30   0.004938    0.005871    0.005311          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
6      trace.from_frontend-2.error_rate         currencyservice-2     30   0.000000    0.000000    0.000000          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
7      trace.from_frontend-2.error_rate   productcatalogservice-2     30   0.000000    0.000000    0.000000          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
8      trace.from_frontend-2.error_rate         currencyservice-1     30   0.000000    0.000000    0.000000          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
9      trace.from_frontend-2.error_rate         currencyservice-0     30   0.000000    0.000000    0.000000          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
10   trace.from_frontend-2.duration_p95   productcatalogservice-2     30   0.004664    0.005864    0.005280          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
11   trace.from_frontend-2.duration_p95   productcatalogservice-1     30   0.004588    0.005850    0.005295          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
12   trace.from_frontend-2.duration_p95   productcatalogservice-0     30   0.004785    0.005813    0.005258          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
13   trace.from_frontend-2.duration_p95         currencyservice-1     30   0.000098    0.000127    0.000113          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
14   trace.from_frontend-2.duration_p95         currencyservice-2     30   0.000103    0.000332    0.000121          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
15     trace.from_frontend-2.error_rate   productcatalogservice-0     30   0.000000    0.000000    0.000000          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
16  trace.from_frontend-1.duration_mean   productcatalogservice-2     30   0.003636    0.004459    0.004044          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
17  trace.from_frontend-1.duration_mean   productcatalogservice-1     30   0.003778    0.004525    0.004105          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
18  trace.from_frontend-1.duration_mean   productcatalogservice-0     30   0.003768    0.004469    0.004062          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800
19  trace.from_frontend-1.duration_mean         currencyservice-2     30   0.000085    0.000129    0.000098          1647808200  2022-03-21 04:30:00 CST+0800        1647809940  2022-03-21 04:59:00 CST+0800, ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0'])```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load or reuse error logs
try:
    error_logs_frontend
except NameError:
    df_err = pd.read_csv('error_logs.csv')
    df_err['timestamp'] = pd.to_numeric(df_err['timestamp'], errors='coerce').astype('Int64')
    df_err['cmdb_id'] = df_err['cmdb_id'].astype(str)
    df_err['message'] = df_err['message'].astype(str)
    # filter frontend and window
    start_ts = 1647808200
    end_ts = 1647809940
    error_logs_frontend = df_err[(df_err['timestamp'] >= start_ts) & (df_err['timestamp'] <= end_ts) & (df_err['cmdb_id'].str.startswith('frontend', na=False))].copy()

# Aggregate frontend errors per-minute (floor to minute)
error_logs_frontend['minute_ts'] = (error_logs_frontend['timestamp'] // 60) * 60

# For earliest/latest message per minute, sort then group
error_logs_frontend_sorted = error_logs_frontend.sort_values(['minute_ts','timestamp'], ascending=[True, True])
frontend_err_minute = (
    error_logs_frontend_sorted
    .groupby('minute_ts', as_index=False)
    .agg(
        frontend_error_count = ('message','count'),
        earliest_timestamp_num = ('timestamp','min'),
        latest_timestamp_num = ('timestamp','max'),
        sample_earliest_message = ('message', 'first'),
        sample_latest_message = ('message', lambda s: s.iloc[-1])
    )
)

# Load or reuse metric.csv
try:
    df_metrics
except NameError:
    df_metrics = pd.read_csv('metric.csv')
    df_metrics['timestamp'] = pd.to_numeric(df_metrics['timestamp'], errors='coerce').astype('Int64')
    df_metrics['value'] = pd.to_numeric(df_metrics['value'], errors='coerce')
    df_metrics['cmdb_id'] = df_metrics['cmdb_id'].astype(str)
    df_metrics['kpi_name'] = df_metrics['kpi_name'].astype(str)

# Incident window
start_ts = 1647808200
end_ts = 1647809940

# Filter adservice service-level rows within window and KPIs of interest
ad_kpis = ['app.grpc.count','app.grpc.sr']
mask_adservice = (df_metrics['cmdb_id'] == 'adservice') & (df_metrics['timestamp'] >= start_ts) & (df_metrics['timestamp'] <= end_ts) & (df_metrics['kpi_name'].isin(ad_kpis))
ad_df = df_metrics[mask_adservice].copy()
ad_df['minute_ts'] = (ad_df['timestamp'] // 60) * 60

# Aggregate per-minute: for app.grpc.count use sum, for app.grpc.sr use mean (service-level)
if not ad_df.empty:
    agg_map = {
        'app.grpc.count': ('value','sum'),
        'app.grpc.sr': ('value','mean')
    }
    # compute per kpi per minute then pivot
    ad_group = ad_df.groupby(['minute_ts','kpi_name'], as_index=False).agg(value_agg=('value','sum'))  # temp sum for all
    # We need correct aggregation per KPI: adjust for sr to mean
    # For simplicity compute separate aggregations
    count_df = ad_df[ad_df['kpi_name']=='app.grpc.count'].groupby('minute_ts', as_index=False).agg(adservice_app_grpc_count=('value','sum'))
    sr_df = ad_df[ad_df['kpi_name']=='app.grpc.sr'].groupby('minute_ts', as_index=False).agg(adservice_app_grpc_sr=('value','mean'))
else:
    count_df = pd.DataFrame(columns=['minute_ts','adservice_app_grpc_count'])
    sr_df = pd.DataFrame(columns=['minute_ts','adservice_app_grpc_sr'])

# Combine adservice per-minute metrics
adservice_per_minute = pd.merge(count_df, sr_df, on='minute_ts', how='outer').sort_values('minute_ts').reset_index(drop=True)

# Left-join frontend error minutes with adservice metrics on minute_ts
correlation_frontend_errors_adservice = pd.merge(
    frontend_err_minute,
    adservice_per_minute,
    left_on='minute_ts',
    right_on='minute_ts',
    how='left'
).sort_values('minute_ts', ascending=True).reset_index(drop=True)

# Keep only rows where frontend_error_count > 0
correlation_frontend_errors_adservice = correlation_frontend_errors_adservice[correlation_frontend_errors_adservice['frontend_error_count'] > 0].copy()

# Prepare display columns and add timestamp_utc8 (minute)
if not correlation_frontend_errors_adservice.empty:
    correlation_frontend_errors_adservice['timestamp'] = correlation_frontend_errors_adservice['minute_ts'].astype('Int64')
    correlation_frontend_errors_adservice['timestamp_utc8'] = pd.to_datetime(correlation_frontend_errors_adservice['minute_ts'].astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    # choose sample_frontend_message as earliest message for the minute
    correlation_frontend_errors_adservice['sample_frontend_message'] = correlation_frontend_errors_adservice['sample_earliest_message']
else:
    # ensure columns exist even if empty
    correlation_frontend_errors_adservice = correlation_frontend_errors_adservice.assign(
        timestamp = pd.Series(dtype='Int64'),
        timestamp_utc8 = pd.Series(dtype='object'),
        sample_frontend_message = pd.Series(dtype='object'),
        adservice_app_grpc_count = pd.Series(dtype='float'),
        adservice_app_grpc_sr = pd.Series(dtype='float')
    )

# Select compact columns and limit to top 50 rows
cols = ['timestamp','timestamp_utc8','frontend_error_count','sample_frontend_message','adservice_app_grpc_count','adservice_app_grpc_sr']
correlation_frontend_errors_adservice = correlation_frontend_errors_adservice[cols].head(50)

# earliest error-minute timestamp (numeric) if any
if not correlation_frontend_errors_adservice.empty:
    earliest_error_minute = int(correlation_frontend_errors_adservice['timestamp'].min())
else:
    earliest_error_minute = None

# Keep correlation_frontend_errors_adservice in kernel as required and return compact table and earliest timestamp
correlation_frontend_errors_adservice, earliest_error_minute

```
Out[7]:
```
Summary (plain English):

- I joined frontend error counts per minute with adservice service-level gRPC metrics for the incident window (2022-03-21 04:30:00 — 04:59:00 UTC+8). The joined table is stored in the kernel variable correlation_frontend_errors_adservice.

- Earliest minute with frontend errors: 1647808200 → 2022-03-21 04:30:00 (UTC+8).

- Scope and frequency:
  - Frontend errors occurred in 29 minutes of the 30-minute window (one minute had no frontend errors).
  - Per-minute frontend error counts are mostly in the mid-teens to mid/upper-20s (examples: 14–27 in the sampled rows).
  - The sample_frontend_message for those minutes is consistently the warning "severity: warning, message: failed to retrieve ads".

- Adservice metrics at the same minutes:
  - adservice_app_grpc_count (per-minute) varies across the window (examples: 35, 57, 69, 72, 73, 99). Peaks to ~99 appear in several minutes.
  - adservice_app_grpc_sr (success rate) is generally high in the sampled minutes (examples: ~62.9 up to ~82.7; many minutes are ~76–81).

- Quick interpretation:
  - Frontend pods repeatedly logged "failed to retrieve ads" starting at 04:30 and persisting almost every minute through the window.
  - The adservice service shows sustained gRPC traffic with moderately high success rates; there is no obvious total service outage in the adservice gRPC success-rate metric.
  - This pattern suggests the problem may not be a complete failure of the adservice service itself (service-level gRPC SR is high), and could instead be related to:
    - frontend → adservice network/mesh path or timeouts,
    - per-pod routing/loadbalancing issues,
    - missing or partial telemetry (some mesh metrics were zero earlier),
    - or frontend-side handling (retries/timeouts) causing "failed to retrieve ads" even though many requests succeed.

- Recommended next steps:
  - Inspect correlation_frontend_errors_adservice (per-minute rows) alongside per-pod adservice metrics and traces for the same timestamps to see whether particular adservice pods or network paths align with error spikes.
  - Check mesh/network telemetry and per-pod logs around the earliest error minute (04:30) to identify packet drops, timeouts, or routing anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp                timestamp_utc8  frontend_error_count                            sample_frontend_message  adservice_app_grpc_count  adservice_app_grpc_sr
0   1647808200  2022-03-21 04:30:00 CST+0800                    19  severity: warning, message: failed to retrieve...                      69.0              72.463768
1   1647808260  2022-03-21 04:31:00 CST+0800                    16  severity: warning, message: failed to retrieve...                      99.0              77.777778
2   1647808320  2022-03-21 04:32:00 CST+0800                    20  severity: warning, message: failed to retrieve...                      57.0              77.192982
3   1647808380  2022-03-21 04:33:00 CST+0800                    15  severity: warning, message: failed to retrieve...                      99.0              77.777778
4   1647808440  2022-03-21 04:34:00 CST+0800                    27  severity: warning, message: failed to retrieve...                      35.0              62.857143
..         ...                           ...                   ...                                                ...                       ...                    ...
24  1647809640  2022-03-21 04:54:00 CST+0800                    18  severity: warning, message: failed to retrieve...                      70.0              81.428571
25  1647809700  2022-03-21 04:55:00 CST+0800                    14  severity: warning, message: failed to retrieve...                      83.0              78.313253
26  1647809760  2022-03-21 04:56:00 CST+0800                    17  severity: warning, message: failed to retrieve...                      72.0              80.555556
27  1647809820  2022-03-21 04:57:00 CST+0800                    14  severity: warning, message: failed to retrieve...                      73.0              76.712329
28  1647809880  2022-03-21 04:58:00 CST+0800                    18  severity: warning, message: failed to retrieve...                      75.0              82.666667

[29 rows x 6 columns], 1647808200)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metrics if present, otherwise load
try:
    df_metrics
except NameError:
    df_metrics = pd.read_csv('metric.csv')

# ensure proper dtypes
df_metrics['timestamp'] = pd.to_numeric(df_metrics['timestamp'], errors='coerce').astype('Int64')
df_metrics['value'] = pd.to_numeric(df_metrics.get('value'), errors='coerce')
df_metrics['cmdb_id'] = df_metrics['cmdb_id'].astype(str)
df_metrics['kpi_name'] = df_metrics['kpi_name'].astype(str)

# target components and KPIs
cmdbs = ['adservice','adservice-0','adservice-1','adservice-2','adservice2-0']
kpis = ['app.grpc.count','app.grpc.sr','app.grpc.mrt']

# Rule 9: compute global stats (using full series BEFORE window filtering)
df_target_full = df_metrics[df_metrics['cmdb_id'].isin(cmdbs) & df_metrics['kpi_name'].isin(kpis)].copy()

global_stats = (
    df_target_full
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        global_count = ('value','count'),
        global_min = ('value','min'),
        global_max = ('value','max'),
        global_mean = ('value','mean'),
        p95_value = ('value', lambda s: float(s.quantile(0.95)) if s.count()>0 else float('nan')),
        p5_value  = ('value', lambda s: float(s.quantile(0.05)) if s.count()>0 else float('nan'))
    )
)

# Incident window
start_ts = 1647808200
end_ts = 1647809940

# Filter rows to window for the same targets
df_window = df_target_full[(df_target_full['timestamp'] >= start_ts) & (df_target_full['timestamp'] <= end_ts)].copy()

# Attach global thresholds to window rows
df_window = df_window.merge(global_stats[['cmdb_id','kpi_name','p95_value','p5_value','global_count']], on=['cmdb_id','kpi_name'], how='left')

# Detect anomalies per rules:
# - for 'app.grpc.sr' : anomaly if value <= P5 (lower-is-bad)
# - for 'app.grpc.count' and 'app.grpc.mrt' : anomaly if value >= P95 (higher-is-bad)
df_window['is_anomaly'] = False
mask_sr = (df_window['kpi_name']=='app.grpc.sr') & df_window['value'].notna() & df_window['p5_value'].notna() & (df_window['value'] <= df_window['p5_value'])
mask_high = df_window['kpi_name'].isin(['app.grpc.count','app.grpc.mrt']) & df_window['value'].notna() & df_window['p95_value'].notna() & (df_window['value'] >= df_window['p95_value'])
df_window.loc[mask_sr | mask_high, 'is_anomaly'] = True

# Store per-point anomaly rows in kernel variable for later inspection
adservice_pod_anomalies = df_window[df_window['is_anomaly']].copy()

# Prepare per-group window aggregates
window_agg = (
    df_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        window_point_count = ('value','count'),
        min_in_window = ('value','min'),
        max_in_window = ('value','max'),
        mean_in_window = ('value','mean')
    )
)

# Per-group anomaly aggregates
anomaly_agg = (
    adservice_pod_anomalies
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        anomalies_count = ('is_anomaly','count'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        latest_anomaly_timestamp = ('timestamp','max')
    )
)

# Combine global, window, and anomaly info
summary = (
    global_stats[['cmdb_id','kpi_name','global_count','p95_value','p5_value']]
    .merge(window_agg, on=['cmdb_id','kpi_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id','kpi_name'], how='left')
)

# Fill NaNs for groups with no window points
summary['window_point_count'] = summary['window_point_count'].fillna(0).astype(int)
summary['anomalies_count'] = summary['anomalies_count'].fillna(0).astype(int)
summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp'].astype('Int64')
summary['latest_anomaly_timestamp'] = summary['latest_anomaly_timestamp'].astype('Int64')
summary['min_in_window'] = summary['min_in_window'].where(pd.notna(summary['min_in_window']), None)
summary['max_in_window'] = summary['max_in_window'].where(pd.notna(summary['max_in_window']), None)
summary['mean_in_window'] = summary['mean_in_window'].where(pd.notna(summary['mean_in_window']), None)

# Keep only groups that have at least one anomaly point in-window
summary_df = summary[summary['anomalies_count'] > 0].copy()

# Order by earliest_anomaly_timestamp ascending and limit to top 20
summary_df = summary_df.sort_values('earliest_anomaly_timestamp', ascending=True).head(20)

# Select and order columns per spec
cols = ['cmdb_id','kpi_name','global_count','p95_value','p5_value','window_point_count','anomalies_count','earliest_anomaly_timestamp','latest_anomaly_timestamp','min_in_window','max_in_window','mean_in_window']
summary_df = summary_df[cols].reset_index(drop=True)

# Display compact summary (do not show raw per-point anomalies)
summary_df

```
Out[8]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def ts_to_str(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')

summary = (
    "Adservice KPI anomaly summary (global stats computed on full series; window = 2022-03-21 04:30:00–04:59:00 UTC+8):\n\n"
    "1) adservice · app.grpc.count\n"
    "- Global samples: 30; P95 = 97.20; P5 = 15.80.\n"
    "- Window: 30 points; min = 13, max = 99, mean ≈ 69.97.\n"
    "- Anomalies (value >= P95): 2 points.\n"
    "- Earliest anomaly: 1647808260 -> " + ts_to_str(1647808260) + "\n"
    "- Latest anomaly:   1647808380 -> " + ts_to_str(1647808380) + "\n\n"
    "2) adservice · app.grpc.sr (success rate)\n"
    "- Global samples: 30; P95 ≈ 82.64; P5 ≈ 66.71.\n"
    "- Window: 30 points; min ≈ 53.85, max ≈ 84.21, mean ≈ 77.61.\n"
    "- Anomalies (value <= P5): 2 points (low success-rate events).\n"
    "- Earliest anomaly: 1647808440 -> " + ts_to_str(1647808440) + "\n"
    "- Latest anomaly:   1647808560 -> " + ts_to_str(1647808560) + "\n\n"
    "3) adservice · app.grpc.mrt (mean response time)\n"
    "- Global samples: 30; P95 ≈ 0.002640 s; P5 ≈ 0.002070 s.\n"
    "- Window: 30 points; min ≈ 0.001822 s, max ≈ 0.002673 s, mean ≈ 0.002475 s.\n"
    "- Anomalies (value >= P95): 2 points.\n"
    "- Earliest anomaly: 1647808920 -> " + ts_to_str(1647808920) + "\n"
    "- Latest anomaly:   1647809160 -> " + ts_to_str(1647809160) + "\n\n"
    "Interpretation & recommended next steps:\n"
    "- Each KPI shows only a small number of anomaly points (2 of 30) within the window, indicating intermittent spikes/dips rather than sustained degradation.\n"
    "- app.grpc.count and app.grpc.mrt anomalies indicate short high-traffic/latency spikes; app.grpc.sr anomalies indicate brief drops in success rate at different minutes.\n"
    "- Correlate these anomaly timestamps with frontend error logs, per-pod metrics, and traces to identify whether specific pods, network/mesh events, or request patterns align with these anomalies.\n"
    "- Because global P95/P5 were computed on the full series, verify whether those thresholds reflect normal operational baselines before treating every flagged point as critical."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id        kpi_name  global_count  p95_value   p5_value  window_point_count  anomalies_count  earliest_anomaly_timestamp  latest_anomaly_timestamp  min_in_window  max_in_window  mean_in_window
0  adservice  app.grpc.count            30   97.20000  15.800000                  30                2                  1647808260                1647808380      13.000000      99.000000       69.966667
1  adservice     app.grpc.sr            30   82.64058  66.714286                  30                2                  1647808440                1647808560      53.846154      84.210526       77.607186
2  adservice    app.grpc.mrt            30    0.00264   0.002070                  30                2                  1647808920                1647809160       0.001822       0.002673        0.002475```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# reuse existing metric dataframe if available
try:
    df_metrics
except NameError:
    df_metrics = pd.read_csv('metric.csv')

# Ensure types
df_metrics['timestamp'] = pd.to_numeric(df_metrics['timestamp'], errors='coerce').astype('Int64')
df_metrics['value'] = pd.to_numeric(df_metrics.get('value'), errors='coerce')
df_metrics['cmdb_id'] = df_metrics['cmdb_id'].astype(str)
df_metrics['kpi_name'] = df_metrics['kpi_name'].astype(str)

# Timezone for any human-readable conversions (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Parameters
start_ts = 1647808200
end_ts = 1647809940
cmdbs = ['adservice-0','adservice-1','adservice-2']
explicit_kpis = ['app.grpc.count','app.grpc.sr','app.grpc.mrt']

# Filter to window and cmdb_ids
mask_window = (df_metrics['timestamp'] >= start_ts) & (df_metrics['timestamp'] <= end_ts)
mask_cmdb = df_metrics['cmdb_id'].isin(cmdbs)
mask_kpi = df_metrics['kpi_name'].isin(explicit_kpis) | df_metrics['kpi_name'].str.contains('mesh|istio', regex=True)

adservice_pod_metrics_window = df_metrics[mask_window & mask_cmdb & mask_kpi].copy()  # kernel variable for later inspection

# Group and compute aggregates
def earliest_timestamp_where(cond_series, ts_series):
    # returns Int64 or pd.NA
    if cond_series.any():
        return int(ts_series[cond_series].min())
    else:
        return pd.NA

grouped = []
grp_cols = ['cmdb_id','kpi_name']
for (cmdb_id, kpi_name), g in adservice_pod_metrics_window.groupby(grp_cols, sort=True):
    cnt = int(g['value'].count())
    vmin = float(g['value'].min()) if not g['value'].isna().all() else None
    vmax = float(g['value'].max()) if not g['value'].isna().all() else None
    vmean = float(g['value'].mean()) if not g['value'].isna().all() else None
    earliest_zero_ts = earliest_timestamp_where(g['value'] == 0, g['timestamp'])
    earliest_pos_ts = earliest_timestamp_where(g['value'] > 0, g['timestamp'])
    grouped.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'count': cnt,
        'min_value': vmin,
        'max_value': vmax,
        'mean_value': vmean,
        'earliest_zero_timestamp': earliest_zero_ts,
        'earliest_positive_timestamp': earliest_pos_ts
    })

summary_df = pd.DataFrame(grouped)

# Sort by cmdb_id then kpi_name and limit to top 20
summary_df = summary_df.sort_values(['cmdb_id','kpi_name'], ascending=[True,True]).reset_index(drop=True).head(20)

# Convert Int64 timestamps to plain ints or None for display consistency
for col in ['earliest_zero_timestamp','earliest_positive_timestamp']:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].astype('Int64')

# Final compact output: summary_df (and adservice_pod_metrics_window is stored in kernel)
summary_df

```
Out[9]:
```
Summary (plain English):

- I filtered metric.csv to the incident window (2022-03-21 04:30:00 — 04:59:00 UTC+8) for adservice pods (adservice-0/1/2) and kept KPIs: app.grpc.*, plus any KPI name containing "mesh" or "istio". The full filtered per-point rows are stored in the kernel variable adservice_pod_metrics_window.

- Output (up to 20 groups shown, sorted by cmdb_id then kpi_name) highlights for adservice-0:
  - All groups have 30 samples (one per minute across the 30-minute window).
  - Many mesh/istio metrics are consistently zero for the whole window (min = max = mean = 0.0) with earliest_zero_timestamp = 1647808200 (2022-03-21 04:30:00). This indicates those mesh counters were zero throughout the window.
  - Several other mesh metrics show substantial non-zero counts/bytes (examples):
    - A mesh.destination.frontend.adservice.istio_request... metric with min 2570, max 14337.5, mean ≈ 9345.17 (earliest positive at 1647808200).
    - Corresponding mesh.destination.frontend.adservice.istio_response... metric with min 4600, max 25875, mean ≈ 16905.0.
    - mesh.destination.unknown.* and mesh.source.* metrics with large numeric ranges (e.g., tens of thousands).
  - For some metrics there is an earliest_positive_timestamp = 1647808200 (service had positive values from the start of the window); for others the earliest_positive_timestamp is later (example: a metric with earliest_positive_timestamp = 1647808200 and earliest_zero_timestamp = 1647808320, indicating zeros and later non-zeros).

- Interpretation and next steps:
  - The coexistence of many zero-valued mesh metrics and many high non-zero mesh counters suggests mixed behaviors across different mesh dimensions (some metrics legitimately zero, others showing heavy traffic). The persistent zeros for many mesh KPIs are suspicious and could indicate telemetry gaps, different label dimensions with no traffic, or instrumentation/collection issues.
  - Because frontends logged repeated "failed to retrieve ads" during this same window, investigate whether the zeroed mesh metrics correspond to the paths used by frontends → adservice (i.e., missing mesh telemetry for those flows), or whether zeros reflect dropped/misrouted requests.
  - Recommend correlating adservice_pod_metrics_window with traces and frontend logs to determine if specific mesh metrics or pods align with the frontend errors and to identify whether the issue is telemetry, mesh routing, or real request failures.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count     min_value     max_value    mean_value  earliest_zero_timestamp  earliest_positive_timestamp
0   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
1   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30   2570.000000  14337.500000   9345.166667                     <NA>                   1647808200
2   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
3   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
4   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      0.009200      0.054125      0.032268                     <NA>                   1647808200
5   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
6   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      4.500000     22.000000     14.666667                     <NA>                   1647808200
7   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
8   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      4.500000     22.000000     14.666667                     <NA>                   1647808200
9   adservice-0  mesh.destination.frontend.adservice.istio_requ...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
10  adservice-0  mesh.destination.frontend.adservice.istio_resp...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
11  adservice-0  mesh.destination.frontend.adservice.istio_resp...     30   4600.000000  25875.000000  16905.000000                     <NA>                   1647808200
12  adservice-0  mesh.destination.frontend.adservice.istio_resp...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
13  adservice-0  mesh.destination.frontend.adservice.istio_resp...     30      0.000000      0.000000      0.000000               1647808200                         <NA>
14  adservice-0  mesh.destination.unknown.adservice.istio_reque...     30   4740.000000   5135.000000   4753.166667                     <NA>                   1647808200
15  adservice-0  mesh.destination.unknown.adservice.istio_reque...     30      0.101325      0.113325      0.104198                     <NA>                   1647808200
16  adservice-0  mesh.destination.unknown.adservice.istio_reque...     30     12.000000     12.000000     12.000000                     <NA>                   1647808200
17  adservice-0  mesh.destination.unknown.adservice.istio_respo...     30  71400.000000  77350.000000  71598.333333                     <NA>                   1647808200
18  adservice-0  mesh.source.adservice.jaeger-collector.istio_r...     30   7000.000000  39375.000000  25725.000000                     <NA>                   1647808200
19  adservice-0  mesh.source.adservice.jaeger-collector.istio_r...     30      0.000000      0.003100      0.001117               1647808320                   1647808200```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metrics if present; otherwise load
try:
    df_metrics
except NameError:
    df_metrics = pd.read_csv('metric.csv')

# Ensure dtypes
df_metrics['timestamp'] = pd.to_numeric(df_metrics['timestamp'], errors='coerce').astype('Int64')
df_metrics['value'] = pd.to_numeric(df_metrics.get('value'), errors='coerce')
df_metrics['cmdb_id'] = df_metrics['cmdb_id'].astype(str)
df_metrics['kpi_name'] = df_metrics['kpi_name'].astype(str)

# Parameters
pods = ['adservice-0','adservice-1','adservice-2']
kpis = ['app.grpc.sr','app.grpc.count']
start_ts = 1647808200
end_ts = 1647809940

# 1) Global statistics across full file for each (cmdb_id, kpi_name)
df_global = df_metrics[df_metrics['cmdb_id'].isin(pods) & df_metrics['kpi_name'].isin(kpis)].copy()

global_stats = (
    df_global
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        global_count = ('value','count'),
        global_min = ('value','min'),
        global_max = ('value','max'),
        global_mean = ('value','mean'),
        p95 = ('value', lambda s: float(s.quantile(0.95)) if s.count()>0 else float('nan')),
        p5  = ('value', lambda s: float(s.quantile(0.05)) if s.count()>0 else float('nan'))
    )
)

# Pivot global stats so we can reference per-pod values easily
g_pivot = global_stats.pivot(index='cmdb_id', columns='kpi_name')
# Flatten columns
g_pivot.columns = ['_'.join(col).strip() for col in g_pivot.columns.values]
g_pivot = g_pivot.reset_index()

# Extract per-pod global p5/p95/count for app.grpc.sr specifically
g_sr = global_stats[global_stats['kpi_name']=='app.grpc.sr'][['cmdb_id','global_count','p95','p5']].rename(columns={
    'global_count':'global_count_app_grpc_sr',
    'p95':'global_p95_app_grpc_sr',
    'p5':'global_p5_app_grpc_sr'
})

# 2) Filter rows to incident window and selected pods/kpis
adservice_pod_appgrpc_window = df_metrics[
    (df_metrics['timestamp'] >= start_ts) &
    (df_metrics['timestamp'] <= end_ts) &
    (df_metrics['cmdb_id'].isin(pods)) &
    (df_metrics['kpi_name'].isin(kpis))
].copy()

# Ensure it's stored in kernel variable as requested
# (variable name: adservice_pod_appgrpc_window)

# 3) For each (cmdb_id,kpi_name) compute window aggregates
window_agg = (
    adservice_pod_appgrpc_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        window_point_count = ('value','count'),
        min_in_window = ('value','min'),
        max_in_window = ('value','max'),
        mean_in_window = ('value','mean')
    )
)

# Pivot window aggregates to get per-pod rows for both KPIs
w_count = window_agg[window_agg['kpi_name']=='app.grpc.count'][['cmdb_id','window_point_count','min_in_window','max_in_window']].rename(columns={
    'window_point_count':'window_point_count_app_grpc_count',
    'min_in_window':'min_app_grpc_count_in_window',
    'max_in_window':'max_app_grpc_count_in_window'
})
w_sr = window_agg[window_agg['kpi_name']=='app.grpc.sr'][['cmdb_id','window_point_count','min_in_window','max_in_window']].rename(columns={
    'window_point_count':'window_point_count_app_grpc_sr',
    'min_in_window':'min_app_grpc_sr_in_window',
    'max_in_window':'max_app_grpc_sr_in_window'
})

# Merge per-pod summary pieces
summary = pd.DataFrame({'cmdb_id': pods}).merge(g_sr, on='cmdb_id', how='left').merge(w_count, on='cmdb_id', how='left').merge(w_sr, on='cmdb_id', how='left')

# Fill NaNs for window counts with 0 and numeric fields as needed
summary['window_point_count_app_grpc_count'] = summary['window_point_count_app_grpc_count'].fillna(0).astype(int)
summary['window_point_count_app_grpc_sr'] = summary['window_point_count_app_grpc_sr'].fillna(0).astype(int)

# 4) For each pod determine earliest in-window timestamp where either:
#    (a) app.grpc.count == 0 OR (b) app.grpc.sr <= global_P5_for_that_pod

# Prepare per-pod p5 for sr
p5_map = g_sr.set_index('cmdb_id')['global_p5_app_grpc_sr'].to_dict()

# Build DataFrame of potential anomaly points from adservice_pod_appgrpc_window
# Condition per-row:
#  - if kpi_name == 'app.grpc.count' and value == 0 -> anomaly
#  - if kpi_name == 'app.grpc.sr' and value <= global_p5 -> anomaly
df_anom_cands = adservice_pod_appgrpc_window.copy()
df_anom_cands['is_anomaly'] = False

# Apply conditions
mask_count_zero = (df_anom_cands['kpi_name']=='app.grpc.count') & (df_anom_cands['value'] == 0)
df_anom_cands.loc[mask_count_zero, 'is_anomaly'] = True

# For sr, compare per-pod p5 (if p5 is NaN, condition will be False)
def sr_anom_mask(row):
    if row['kpi_name'] != 'app.grpc.sr' or pd.isna(row['value']):
        return False
    p5 = p5_map.get(row['cmdb_id'], None)
    if p5 is None or pd.isna(p5):
        return False
    return row['value'] <= p5

mask_sr_anom = df_anom_cands.apply(sr_anom_mask, axis=1)
df_anom_cands.loc[mask_sr_anom, 'is_anomaly'] = True

# Store per-point anomalies in kernel variable adservice_pod_anomalies (as requested earlier in other tasks)
adservice_pod_anomalies = df_anom_cands[df_anom_cands['is_anomaly']].copy()

# For each pod, find earliest anomaly timestamp if any
earliest_anoms = (
    adservice_pod_anomalies.groupby('cmdb_id', as_index=False)
    .agg(earliest_anomaly_timestamp=('timestamp','min'))
)

# Merge earliest anomalies into summary
summary = summary.merge(earliest_anoms, on='cmdb_id', how='left')

# Convert earliest timestamp to Int64 and to UTC+8 string where present
summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp'].astype('Int64')
summary['earliest_anomaly_timestamp_utc8'] = summary['earliest_anomaly_timestamp'].apply(
    lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z') if pd.notna(t) else None
)

# Reorder and select requested columns
out_cols = [
    'cmdb_id',
    'global_count_app_grpc_sr',
    'global_p5_app_grpc_sr',
    'global_p95_app_grpc_sr',
    'window_point_count_app_grpc_count',
    'min_app_grpc_count_in_window',
    'max_app_grpc_count_in_window',
    'window_point_count_app_grpc_sr',
    'min_app_grpc_sr_in_window',
    'max_app_grpc_sr_in_window',
    'earliest_anomaly_timestamp',
    'earliest_anomaly_timestamp_utc8'
]
# Some columns may be missing if no global stats existed; ensure they exist
for c in out_cols:
    if c not in summary.columns:
        summary[c] = pd.NA

summary_final = summary[out_cols].reset_index(drop=True).head(10)

# Keep adservice_pod_appgrpc_window and adservice_pod_anomalies in kernel for later inspection
adservice_pod_appgrpc_window, adservice_pod_anomalies, summary_final

```
Out[10]:
```
Summary (plain English):

- I filtered metric.csv for the three pods (adservice-0, adservice-1, adservice-2) and the KPIs app.grpc.sr and app.grpc.count, then computed global and in-window stats and searched for anomalies per your rules.

- Result: no matching metric rows were found for those pod-level combinations in the file:
  - The in-window dataset adservice_pod_appgrpc_window is empty.
  - No per-point anomalies were detected (adservice_pod_anomalies is empty).
  - The per-pod summary contains no numeric global or window statistics (all fields are empty/NaN) and earliest_anomaly_timestamp is null for each pod.

- Interpretation and next steps:
  - It appears app.grpc.* metrics for gRPC success-rate/count exist at the service level (adservice) in this dataset but not under the pod names adservice-0/1/2 — which matches prior observations where app.grpc metrics were present for 'adservice' but not the pods.
  - To proceed, either:
    - Run the same aggregation for cmdb_id 'adservice' (service-level), or
    - Inspect metric.csv to list available cmdb_id and kpi_name combinations (e.g., check unique kpi_name values for adservice pods) to find the correct per-pod KPI names.
  - I can run either of those follow-ups if you want.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp, cmdb_id, kpi_name, value]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, kpi_name, value, is_anomaly]
Index: [],        cmdb_id  global_count_app_grpc_sr  global_p5_app_grpc_sr  global_p95_app_grpc_sr  window_point_count_app_grpc_count  min_app_grpc_count_in_window  max_app_grpc_count_in_window  window_point_count_app_grpc_sr  min_app_grpc_sr_in_window  max_app_grpc_sr_in_window  earliest_anomaly_timestamp earliest_anomaly_timestamp_utc8
0  adservice-0                       NaN                    NaN                     NaN                                  0                           NaN                           NaN                               0                        NaN                        NaN                        <NA>                            None
1  adservice-1                       NaN                    NaN                     NaN                                  0                           NaN                           NaN                               0                        NaN                        NaN                        <NA>                            None
2  adservice-2                       NaN                    NaN                     NaN                                  0                           NaN                           NaN                               0                        NaN                        NaN                        <NA>                            None)```
```